In [28]:
# CELL 14: Build monthly feature vectors per customer (training period only)
# 4.1 Feature Engineering with explicit λ and half-life reporting

train_df = df_traj[df_traj['purchase_month'].isin(train_months)].copy()

# Compute λ for target half-life of 45 days
LAMBDA_TARGET = np.log(2) / 45
print(f"Re-tuned λ: {LAMBDA_TARGET:.6f} (half-life = 45 days)")

# Reference date: last day of training period
reference_date = train_df['InvoiceDate'].max()
print(f"Reference date for recency: {reference_date}")

# Feature engineering per customer per month
def compute_monthly_features(df, month, reference_date, lambda_val):
    month_data = df[df['purchase_month'] == month]
    
    features = month_data.groupby('Customer ID').agg(
        frequency=('Invoice', 'nunique'),
        monetary=('TotalPrice', 'sum'),
        avg_order_value=('TotalPrice', 'mean'),
        quantity=('Quantity', 'sum'),
        last_purchase=('InvoiceDate', 'max')
    ).reset_index()
    
    # Recency: days since last purchase from reference date
    features['recency'] = (reference_date - features['last_purchase']).dt.days
    
    # Time-decayed frequency
    decay_weights = np.exp(-lambda_val * (reference_date - features['last_purchase']).dt.days)
    features['decayed_frequency'] = features['frequency'] * decay_weights
    
    features['month'] = month
    features.drop(columns=['last_purchase'], inplace=True)
    
    return features

# Build monthly sequences
monthly_snapshots = []
for month in train_months:
    snap = compute_monthly_features(train_df, month, reference_date, LAMBDA_TARGET)
    monthly_snapshots.append(snap)

# Create sequence tensor
# Shape: (customers, time_steps, features)
feature_cols = ['recency', 'frequency', 'monetary', 'avg_order_value', 'quantity', 'decayed_frequency']

# Build customer x month matrix
customer_ids = sorted(eligible_customers)
n_customers = len(customer_ids)
n_months = len(train_months)
n_features = len(feature_cols)

sequence_data = np.zeros((n_customers, n_months, n_features))
mask = np.zeros((n_customers, n_months), dtype=bool)

customer_to_idx = {cid: i for i, cid in enumerate(customer_ids)}
month_to_idx = {m: i for i, m in enumerate(train_months)}

for snap, month in zip(monthly_snapshots, train_months):
    m_idx = month_to_idx[month]
    for _, row in snap.iterrows():
        cid = str(row['Customer ID'])
        if cid in customer_to_idx:
            c_idx = customer_to_idx[cid]
            sequence_data[c_idx, m_idx, :] = row[feature_cols].values
            mask[c_idx, m_idx] = True

print(f"Sequence data shape: {sequence_data.shape}")
print(f"Active (non-zero) cells: {mask.sum()} / {mask.size} ({mask.sum()/mask.size*100:.1f}%)")

Re-tuned λ: 0.015403 (half-life = 45 days)
Reference date for recency: 2011-09-30 15:33:00
Sequence data shape: (3030, 22, 6)
Active (non-zero) cells: 18818 / 66660 (28.2%)


In [29]:
# CELL 15: Scale features
scaler = StandardScaler()
# Fit on all non-zero entries across all time steps
valid_data = sequence_data[mask]
scaler.fit(valid_data)

scaled_sequence = np.zeros_like(sequence_data)
scaled_sequence[mask] = scaler.transform(valid_data)

print("Feature means after scaling (should be ~0):", scaled_sequence[mask].mean(axis=0).round(3))
print("Feature stds after scaling (should be ~1):", scaled_sequence[mask].std(axis=0).round(3))

Feature means after scaling (should be ~0): [ 0. -0. -0.  0. -0. -0.]
Feature stds after scaling (should be ~1): [1. 1. 1. 1. 1. 1.]


In [30]:
# CELL 16: Convert to PyTorch tensors
device = torch.device('cpu')  # CPU is sufficient as discussed

X = torch.FloatTensor(scaled_sequence).to(device)
M = torch.FloatTensor(mask.astype(float)).to(device)  # 1=active, 0=masked

print(f"X tensor shape: {X.shape}")
print(f"M tensor shape: {M.shape}")

X tensor shape: torch.Size([3030, 22, 6])
M tensor shape: torch.Size([3030, 22])
